In [1]:
# Imports & constants
from pathlib import Path
import re
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from xgboost import XGBClassifier
import joblib

SEED = 42
DATA_DIR = Path('../data/processed')
# only keep LPPT files (same dataset as original)
all_npy = sorted([p for p in DATA_DIR.glob('*.npy') if 'LPPT' in p.name], key=lambda p: p.name)
name2path = {p.name: p for p in all_npy}
print(f'Found {len(all_npy)} LPPT .npy files in: {DATA_DIR.resolve()}')

Found 20 LPPT .npy files in: D:\ISCF\Early-Fault-Detection-in-PV-A-Case-Study-on-the-GPVS-Faults-Dataset\data\processed


In [2]:
# Pairing logic: for each X_*.npy create corresponding Y_*.npy by prefix replacement
import warnings
def find_xy_pairs(name2path):
    pairs = []
    for name, p in name2path.items():
        if not name.startswith('X_') or 'LPPT' not in name:
            continue
        yname = name.replace('X_', 'Y_', 1)
        ypath = name2path.get(yname)
        if ypath is None:
            warnings.warn(f'Missing label for {name}: expected {yname} - skipping')
            continue
        pairs.append((p, ypath))
    # deterministic order by X filename
    pairs = sorted(pairs, key=lambda t: t[0].name)
    return pairs

pairs = find_xy_pairs(name2path)
print(f'Found {len(pairs)} X/Y pairs (will load and concatenate samples)')

Found 10 X/Y pairs (will load and concatenate samples)


In [3]:
# Load samples from paired files. Each file may contain multiple samples; we preserve in-file order.
def load_samples_from_pairs(pairs):
    X_list, Y_list = [], []
    for xp, yp in pairs:
        X_arr = np.load(xp, allow_pickle=True)
        Y_arr = np.load(yp, allow_pickle=True)
        if len(X_arr) != len(Y_arr):
            raise ValueError(f'Length mismatch for {xp.name} / {yp.name}: {len(X_arr)} vs {len(Y_arr)}')
        X_list.extend(list(X_arr))
        Y_list.extend(list(Y_arr))
    return np.array(X_list), np.array(Y_list)

X_all, Y_all = load_samples_from_pairs(pairs)
print('Loaded samples:', X_all.shape, Y_all.shape)
# ensure numeric dtype and expected shape (N,T,C)
X_all = np.asarray(X_all, dtype=np.float32)


Loaded samples: (72791, 200, 13) (72791,)


In [4]:
# Train/val/test split: stratified, deterministic, 60/20/20
# Convert labels to strings first, then stratify
Y_str = np.array([str(y) for y in Y_all])
# first split train vs temp(40%)
X_train, X_temp, y_train, y_temp = train_test_split(X_all, Y_str, train_size=0.6, stratify=Y_str, random_state=SEED, shuffle=True)
# split temp into val/test equal halves => each 20% overall
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=SEED, shuffle=True)
print('shapes:')
print(' X_train', X_train.shape, 'y_train', np.unique(y_train, return_counts=True))
print(' X_val  ', X_val.shape,  'y_val', np.unique(y_val, return_counts=True))
print(' X_test ', X_test.shape, 'y_test', np.unique(y_test, return_counts=True))

shapes:
 X_train (43674, 200, 13) y_train (array(['F0L', 'F1L', 'F2L', 'F3L', 'F4L', 'F5L', 'F6L', 'F7L'],
      dtype='<U3'), array([5741, 5153, 5677, 4132, 5753, 5713, 5753, 5752]))
 X_val   (14558, 200, 13) y_val (array(['F0L', 'F1L', 'F2L', 'F3L', 'F4L', 'F5L', 'F6L', 'F7L'],
      dtype='<U3'), array([1913, 1718, 1893, 1378, 1917, 1904, 1917, 1918]))
 X_test  (14559, 200, 13) y_test (array(['F0L', 'F1L', 'F2L', 'F3L', 'F4L', 'F5L', 'F6L', 'F7L'],
      dtype='<U3'), array([1914, 1717, 1892, 1377, 1918, 1905, 1918, 1918]))


In [5]:
# Alternate, compact feature extractor (per-channel summary + simple spectral stats)
from scipy.stats import skew, kurtosis
def extract_features_compact(X, eps=1e-12):
    # X: (N, T, C)
    X = np.asarray(X, dtype=np.float32)
    assert X.ndim == 3, f'expected (N,T,C), got {X.shape}'
    N, T, C = X.shape
    feats_ch = []
    names = []
    for c in range(C):
        x = X[:, :, c]  # (N, T)
        mu = x.mean(axis=1)
        sd = x.std(axis=1) + eps
        mn = x.min(axis=1)
        mx = x.max(axis=1)
        med = np.median(x, axis=1)
        iqr = np.quantile(x, 0.75, axis=1) - np.quantile(x, 0.25, axis=1)
        sk = skew(x, axis=1, bias=False)
        kt = kurtosis(x, axis=1, fisher=True, bias=False)
        rms = np.sqrt((x**2).mean(axis=1) + eps)
        ptp = mx - mn
        zc = ((x[:, 1:] * x[:, :-1]) < 0).mean(axis=1)
        # simple lag-1 autocorr normalized
        xm = x - mu[:, None]
        ac1 = (xm[:, :-1] * xm[:, 1:]).mean(axis=1) / (sd**2 + eps)
        # rfft power bands (three bands: low/mid/high fractions of Nyquist)
        fft = np.fft.rfft(xm, axis=1)
        power = (fft.real**2 + fft.imag**2)
        power[:, 0] = 0.0  # remove DC
        psum = power.sum(axis=1) + eps
        freqs = np.fft.rfftfreq(T, d=1.0)
        # define relative bands by fraction of Nyquist (0..0.5)
        f = freqs / freqs.max() if freqs.max() > 0 else freqs
        low_mask = f <= 0.2
        mid_mask = (f > 0.2) & (f <= 0.6)
        high_mask = f > 0.6
        bp_low = power[:, low_mask].sum(axis=1) / psum
        bp_mid = power[:, mid_mask].sum(axis=1) / psum
        bp_high = power[:, high_mask].sum(axis=1) / psum
        ch_feats = np.stack([mu, sd, mn, mx, med, iqr, sk, kt, rms, ptp, zc, ac1, bp_low, bp_mid, bp_high], axis=1)
        feats_ch.append(ch_feats)
        names.extend([f'ch{c}_mean', f'ch{c}_std', f'ch{c}_min', f'ch{c}_max', f'ch{c}_median', f'ch{c}_iqr', f'ch{c}_skew', f'ch{c}_kurt', f'ch{c}_rms', f'ch{c}_ptp', f'ch{c}_zcr', f'ch{c}_ac1', f'ch{c}_bp_low', f'ch{c}_bp_mid', f'ch{c}_bp_high'])
    feats = np.concatenate(feats_ch, axis=1).astype(np.float32)
    return feats, names

# build features for splits
X_train_feat, feat_names = extract_features_compact(X_train)
X_val_feat, _ = extract_features_compact(X_val)
X_test_feat, _ = extract_features_compact(X_test)
print('Feature shapes:', X_train_feat.shape, X_val_feat.shape, X_test_feat.shape)
le = LabelEncoder()
y_train_int = le.fit_transform(y_train)
y_val_int = le.transform(y_val)
y_test_int = le.transform(y_test)
print('Classes:', list(le.classes_))

C:\Users\KIIT\AppData\Local\Temp\ipykernel_14148\584086342.py:18: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  sk = skew(x, axis=1, bias=False)
C:\Users\KIIT\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\scipy\stats\_stats_py.py:1361: RuntimeWarning: invalid value encountered in divide
  nval = ((n - 1.0) * n)**0.5 / (n - 2.0) * m3 / m2**1.5
C:\Users\KIIT\AppData\Local\Temp\ipykernel_14148\584086342.py:19: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  kt = kurtosis(x, axis=1, fisher=True, bias=False)
C:\Users\KIIT\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\scipy\stats\_stats_py.p

Feature shapes: (43674, 195) (14558, 195) (14559, 195)
Classes: [np.str_('F0L'), np.str_('F1L'), np.str_('F2L'), np.str_('F3L'), np.str_('F4L'), np.str_('F5L'), np.str_('F6L'), np.str_('F7L')]


In [6]:
# Hyperparameter search on X_train using StratifiedKFold; then retrain on train+val
base = XGBClassifier(objective='multi:softprob', num_class=len(le.classes_), tree_method='hist', eval_metric='mlogloss', random_state=SEED, n_jobs=-1)
param_dist = {
    'n_estimators': [200, 400, 800],
    'max_depth': [3, 6, 9],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.7, 0.9, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'reg_alpha': [0.0, 1e-2],
    'reg_lambda': [1.0, 2.0],
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
search = RandomizedSearchCV(base, param_distributions=param_dist, n_iter=30, scoring='f1_macro', cv=cv, random_state=SEED, n_jobs=-1, verbose=2)
search.fit(X_train_feat, y_train_int)
print('Best CV on TRAIN folds (f1_macro):', search.best_score_)
best_params = search.best_params_
print('Best params:', best_params)

Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best CV on TRAIN folds (f1_macro): 1.0
Best params: {'subsample': 0.7, 'reg_lambda': 2.0, 'reg_alpha': 0.0, 'n_estimators': 800, 'max_depth': 9, 'learning_rate': 0.05, 'colsample_bytree': 1.0}


In [7]:
# Final model: retrain on train+val (to use more data), evaluate on test
X_trainval_feat = np.vstack([X_train_feat, X_val_feat])
y_trainval = np.concatenate([y_train_int, y_val_int])
final_model = XGBClassifier(**best_params, objective='multi:softprob', num_class=len(le.classes_), tree_method='hist', eval_metric='mlogloss', random_state=SEED, n_jobs=-1)
final_model.fit(X_trainval_feat, y_trainval)
y_pred = final_model.predict(X_test_feat)
print('Test accuracy:', accuracy_score(y_test_int, y_pred))
print('\nPer-class report on TEST:')
print(classification_report(y_test_int, y_pred, target_names=list(le.classes_), digits=4))
print('Confusion matrix (rows=true, cols=pred):')
print(confusion_matrix(y_test_int, y_pred))

# save model and feature names for later use
joblib.dump({'model': final_model, 'feat_names': feat_names, 'label_encoder': le}, 'xgb_lppt_model.joblib')
print('Saved final model -> xgb_lppt_model.joblib')

Test accuracy: 1.0

Per-class report on TEST:
              precision    recall  f1-score   support

         F0L     1.0000    1.0000    1.0000      1914
         F1L     1.0000    1.0000    1.0000      1717
         F2L     1.0000    1.0000    1.0000      1892
         F3L     1.0000    1.0000    1.0000      1377
         F4L     1.0000    1.0000    1.0000      1918
         F5L     1.0000    1.0000    1.0000      1905
         F6L     1.0000    1.0000    1.0000      1918
         F7L     1.0000    1.0000    1.0000      1918

    accuracy                         1.0000     14559
   macro avg     1.0000    1.0000    1.0000     14559
weighted avg     1.0000    1.0000    1.0000     14559

Confusion matrix (rows=true, cols=pred):
[[1914    0    0    0    0    0    0    0]
 [   0 1717    0    0    0    0    0    0]
 [   0    0 1892    0    0    0    0    0]
 [   0    0    0 1377    0    0    0    0]
 [   0    0    0    0 1918    0    0    0]
 [   0    0    0    0    0 1905    0    0]
 [  

In [8]:
# Feature importance (gain) mapped to human names
booster = final_model.get_booster()
imp = booster.get_score(importance_type='gain')
items = sorted(imp.items(), key=lambda x: x[1], reverse=True)
topk = min(30, len(items))
for k, v in items[:topk]:
    idx = int(k[1:])
    print(f'{idx:4d}  {feat_names[idx]:30s}  gain={v:.6f}')

# optional: plot
# from xgboost import plot_importance
# import matplotlib.pyplot as plt
# plot_importance(final_model, importance_type='gain', max_num_features=30)
# plt.show()

   4  ch0_median                      gain=1032.849854
  79  ch5_median                      gain=371.819214
   8  ch0_rms                         gain=368.524719
  33  ch2_max                         gain=342.028717
 189  ch12_ptp                        gain=334.479004
  49  ch3_median                      gain=264.285736
  15  ch1_mean                        gain=249.632019
 144  ch9_ptp                         gain=239.244507
  23  ch1_rms                         gain=200.107285
  81  ch5_skew                        gain=136.651566
  51  ch3_skew                        gain=95.321251
  46  ch3_std                         gain=89.985016
   2  ch0_min                         gain=45.086891
  60  ch4_mean                        gain=43.066387
 181  ch12_std                        gain=39.038834
   0  ch0_mean                        gain=38.181480
 165  ch11_mean                       gain=33.422226
 174  ch11_ptp                        gain=20.427462
  20  ch1_iqr                      